<a href="https://colab.research.google.com/github/karankiyaswanth/exp-6-irs/blob/main/Exp6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urljoin
from collections import deque

def is_allowed(url, allowed_domains):
    domain = urlparse(url).netloc
    return any(domain.endswith(ad) for ad in allowed_domains)

def crawl_news(topic, allowed_domains, max_pages=20):
    # Seed URLs
    seeds = [f"https://{domain}" for domain in allowed_domains]

    visited = set()
    queue = deque(seeds)
    results = []

    # Use a standard user-agent to prevent websites from blocking our requests
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
    }

    while queue and len(visited) < max_pages:
        url = queue.popleft()
        if url in visited:
            continue
        try:
            response = requests.get(url, headers=headers, timeout=5)
            if response.status_code != 200:
                continue
            html = response.text
            soup = BeautifulSoup(html, "html.parser")

            # Check if topic is in page text
            page_text = soup.get_text()
            if topic.lower() in page_text.lower():
                title = soup.title.string if soup.title and soup.title.string else 'No Title'
                results.append((title.strip(), url))
                print(f"Found: {title.strip()} - {url}")

            # Extract and enqueue links
            for link_tag in soup.find_all('a', href=True):
                link = urljoin(url, link_tag['href'])
                # Normalize link (remove fragments, query params)
                link_parsed = urlparse(link)
                link = link_parsed.scheme + "://" + link_parsed.netloc + link_parsed.path
                if is_allowed(link, allowed_domains) and link not in visited:
                    queue.append(link)

            visited.add(url)
        except Exception as e:
            # Handle errors like timeouts or connection issues gracefully
            continue

    return results

if __name__ == "__main__":
    # User inputs with default fallback values for easier execution
    topic = input("Enter the topic to search for (e.g. technology): ").strip() or "technology"
    print("Enter allowed websites (comma-separated, e.g. cnn.com,bbc.com):")
    allowed_input = input().strip() or "cnn.com,bbc.com"
    allowed_domains = [d.strip() for d in allowed_input.split(",") if d.strip()]

    max_pages_input = input("Enter max number of pages to crawl (default 20): ").strip()
    max_pages = int(max_pages_input) if max_pages_input.isdigit() else 20

    print(f"\nStarting crawl for topic '{topic}' on domains: {allowed_domains}\n")
    results = crawl_news(topic, allowed_domains, max_pages)

    print("\n--- Crawl Results ---")
    if results:
        for i, (title, url) in enumerate(results, 1):
            print(f"{i}. {title}\n   {url}\n")
    else:
        print("No matching pages found.")

Enter the topic to search for (e.g. technology): climate change
Enter allowed websites (comma-separated, e.g. cnn.com,bbc.com):
cnn.com, bbc.com
Enter max number of pages to crawl (default 20): 10

Starting crawl for topic 'climate change' on domains: ['cnn.com', 'bbc.com']

Found: BBC Home - Breaking News, World News, US News, Sports, Business, Innovation, Climate, Culture, Travel, Video & Audio - https://bbc.com

--- Crawl Results ---
1. BBC Home - Breaking News, World News, US News, Sports, Business, Innovation, Climate, Culture, Travel, Video & Audio
   https://bbc.com

